In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta

# -------------------------------------------------------------------
# Config
# -------------------------------------------------------------------
roles        = ["Patient", "Radiation Oncology", "Urologic Oncology"]
races        = ["Black or African American", "White", "Asian",
                "American Indian or Alaska Native"]
next_steps   = ["Still deciding", "Further workup", "Active surveillance",
                "Radiation", "Surgery"]

num_patients = 100000          # change if you like
num_questions = 10

# -------------------------------------------------------------------
# Synthetic-data generator
# -------------------------------------------------------------------
records = []
for i in range(1, num_patients + 1):
    base_index = f"{i:03d}"        # e.g. "001", "002", …
    treatment_id = f"T{base_index}"

    # Patient-specific attributes (age, race, distance) –
    # copied into the patient record only
    age       = random.randint(50, 84)
    race      = random.choice(races)
    distance  = random.randint(1, 200)
    next_step = random.choice(next_steps)
    appt_date = (datetime.today() -
                 timedelta(days=random.randint(0, 364))).strftime("%Y-%m-%d")

    for role in roles:
        # Role-prefixed ID
        prefix = "P" if role == "Patient" else ("R" if role.startswith("Radiation") else "U")
        record_id = f"{prefix}{base_index}"

        # Survey answers
        answers = [random.randint(1, 6) for _ in range(num_questions)]

        record = {
            "id": record_id,            #  <-- renamed field
            "treatment_id": treatment_id,
            "submission_role": role,
            "appointment_date": appt_date,
            "next_steps": next_step,
            "score_below_3": any(a <= 3 for a in answers),
            # Patient-only demographics
            "age":  age  if role == "Patient" else None,
            "race": race if role == "Patient" else None,
            "distance_miles": distance if role == "Patient" else None,
        }

        # Add survey_q1 … survey_q10
        record.update({f"survey_q{j}": val
                       for j, val in enumerate(answers, start=1)})

        records.append(record)

df = pd.DataFrame(records)
df.to_csv("radar_survey_synthetic_data.csv", index=False)
print("✔ Synthetic data saved to radar_survey_synthetic_data.csv")
